In [1]:
from ast import literal_eval  # as tuple literal_eval -> convert string representation of list to actual list
from pathlib import Path

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display


In [2]:
# --- par‑plant ---------------------------------------------------------------
path_box = Path("/home/loai/Documents/code/RSMLExtraction/Results/Reconstruction/measure_per_plant.csv")
df_plant_wid = pd.read_csv(path_box)
df_plant_wid['root_ids'] = df_plant_wid['root_ids'].apply(literal_eval)
df_plant_wid

,model,split,box,metric,time,root_ids,source,value
0,Segformer_bce_dice,Val,230629PN031,NumberOfOrgans,1,"(1, 1, 1)",Prediction,1.000000
1,Segformer_bce_dice,Val,230629PN031,NumberOfOrgans,1,"(1, 1, 1)",expertized,1.000000
2,Segformer_bce_dice,Val,230629PN031,NumberOfOrgans,1,"(1, 1, 1)",before_expertized,1.000000
3,Segformer_bce_dice,Val,230629PN031,NumberOfOrgans,1,"(36, 47, 45)",Prediction,1.000000
4,Segformer_bce_dice,Val,230629PN031,NumberOfOrgans,1,"(36, 47, 45)",expertized,1.000000
...,...,...,...,...,...,...,...,...
209491,Unet_cldice_dice,Test,230629PN018,LateralRootLength,29,"(10, 14, 8)",expertized,125.584577
209492,Unet_cldice_dice,Test,230629PN018,LateralRootLength,29,"(10, 14, 8)",before_expertized,20.126268
209493,Unet_cldice_dice,Test,230629PN018,LateralRootLength,29,"(17, 23, 13)",Prediction,4.000000
209494,Unet_cldice_dice,Test,230629PN018,LateralRootLength,29,"(17, 23, 13)",expertized,119.834592


In [3]:
df_grouped = (
    df_plant_wid
    .set_index(['model', 'split', 'box', 'metric', 'root_ids', 'source', 'time'])
    .sort_index(level='time')
)
df_grouped

value
model         split box         metric           root_ids     source            time             
Segformer_bce Test  230629PN008 Convex_Area_Hull (8, 1, 1)    Prediction        1      330.901647
                                                              before_expertized 1      328.799340
                                                              expertized        1      328.799340
                                                 (16, 11, 8)  Prediction        1      406.132539
                                                              before_expertized 1      406.477047
...                                                                                           ...
Unet_dice     Val   230629PN031 TotalRootLength  (68, 66, 63) before_expertized 29    2846.313710
                                                              expertized        29    2846.313710
                                                 (86, 84, 81) Prediction        29    3180.436231
                                                              before_expertized 29    3313.831728
                                                              expertized        29    3296.533489

[209496 rows x 1 columns]

In [4]:
models = df_grouped.index.get_level_values('model').unique()
splits = df_grouped.index.get_level_values('split').unique()
boxes = df_grouped.index.get_level_values('box').unique()
metrics = df_grouped.index.get_level_values('metric').unique()

print("Models disponibles :", models)
print("Splits disponibles :", splits)
print("Boxes disponibles  :", boxes)
print("Metrics disponibles:", metrics)

Models disponibles : Index(['Segformer_bce', 'Segformer_bce_dice', 'Segformer_dice', 'Unet_bce',
       'Unet_bce_dice', 'Unet_cldice_dice', 'Unet_dice'],
      dtype='object', name='model')
Splits disponibles : Index(['Test', 'Val'], dtype='object', name='split')
Boxes disponibles  : Index(['230629PN008', '230629PN010', '230629PN018', '230629PN024',
       '230629PN027', '230629PN006', '230629PN012', '230629PN014',
       '230629PN019', '230629PN031'],
      dtype='object', name='box')
Metrics disponibles: Index(['Convex_Area_Hull', 'Intercept_curve_Area', 'LateralRootLength',
       'NumberOfLateralRoots', 'NumberOfOrgans', 'PrimaryRootLength',
       'RootDensity', 'TotalRootLength'],
      dtype='object', name='metric')


In [5]:
def get_root_ids(model, split, box, metric):
    return (
        df_grouped
        .xs((model, split, box, metric), level=('model', 'split', 'box', 'metric'))
        .index.get_level_values('root_ids')
        .unique()
    )


def get_sources(model, split, box, metric, root):
    return (
        df_grouped
        .xs((model, split, box, metric, root), level=('model', 'split', 'box', 'metric', 'root_ids'))
        .index.get_level_values('source')
        .unique()
    )


def plot_root(model_idx, split_idx, box_idx, metric_idx, root_idx):
    model = models[model_idx]
    split = splits[split_idx]
    box = boxes[box_idx + split_idx * 5]
    metric = metrics[metric_idx]
    roots = get_root_ids(model, split, box, metric)
    root = roots[root_idx]

    df_tmp = (
        df_grouped
        .xs((model, split, box, metric, root),
            level=('model', 'split', 'box', 'metric', 'root_ids'))
        .reset_index()
    )
    #print(df_tmp.head())

    # Map styles + offset pour le jitter en abscisse
    style_map = {
        'before_expertized': dict(color='orange', linestyle='--', marker=None,
                                  linewidth=2.5, markersize=8, alpha=0.8, xoff=-0.1),
        'expertized': dict(color='green', linestyle='-', marker='o',
                           linewidth=2.5, markersize=8, alpha=0.8, xoff=0.0),
        'Prediction': dict(color='red', linestyle='-', marker='x',
                           linewidth=2.5, markersize=8, alpha=0.8, xoff=0.1),
    }

    plt.figure(figsize=(16, 8))
    for source, style in style_map.items():
        df_s = df_tmp[df_tmp['source'] == source]
        if df_s.empty:
            continue

        # convertir time en float si nécessaire
        x = pd.to_numeric(df_s['time'], errors='coerce').astype(float)
        # appliquer le jitter
        x = x + style['xoff']

        plt.plot(
            x, df_s['value'],
            label=source,
            color=style['color'],
            linestyle=style['linestyle'],
            marker=style['marker'],
            linewidth=style['linewidth'],
            markersize=style['markersize'],
            alpha=style['alpha']
        )

        # annoter le dernier point
        last_x = x.iloc[-1]
        last_y = df_s['value'].iloc[-1]
        plt.text(
            last_x, last_y,
            f" {source}",
            va='center',
            fontsize=9,
            color=style['color']
        )

    # Remettre les vraies valeurs de time en ticks
    uniq_times = sorted(df_tmp['time'].unique())
    plt.xticks(uniq_times, uniq_times)

    plt.title(f"{metric} – {model} / {split} / {box} / root {root}")
    plt.xlabel('Temps')
    plt.ylabel('Value')
    plt.grid(True, linestyle=':', alpha=0.5)
    plt.legend(title='Source')
    plt.tight_layout()
    plt.show()

In [ ]:
model_slider = widgets.IntSlider(min=0, max=len(models) - 1, step=1, value=0, description='Model')
split_slider = widgets.IntSlider(min=0, max=len(splits) - 1, step=1, value=0, description='Split')
box_slider = widgets.IntSlider(min=0, max=4, step=1, value=0, description='Box')
metric_slider = widgets.IntSlider(min=0, max=len(metrics) - 1, step=1, value=0, description='Metric')

# Slider “root” : on fixe temporairement sa plage max à 0…9 – on mettra à jour dynamiquement
root_slider = widgets.IntSlider(min=0, max=0, step=1, value=0, description='Root')


# Callback pour mettre à jour le slider root_ids quand on change l’une des 4 premières valeurs
def update_root_slider(*args):
    roots = get_root_ids(
        models[model_slider.value],
        splits[split_slider.value],
        boxes[box_slider.value + split_slider.value * 5],
        metrics[metric_slider.value]
    )
    root_slider.max = len(roots) - 1
    if (root_slider.value >= root_slider.max):
        root_slider.value = root_slider.max


# On observe les changements
for w in (model_slider, split_slider, box_slider, metric_slider):
    w.observe(update_root_slider, 'value')

# Lancer la première initialisation
update_root_slider()

# 6) Affichage interactif
ui = widgets.HBox([model_slider, split_slider, box_slider, metric_slider, root_slider])
out = widgets.interactive_output(
    plot_root,
    {
        'model_idx': model_slider,
        'split_idx': split_slider,
        'box_idx': box_slider,
        'metric_idx': metric_slider,
        'root_idx': root_slider
    }
)

display(ui, out)

Output()

## Machin

In [7]:
df_plant_wid

,model,split,box,metric,time,root_ids,source,value
0,Segformer_bce_dice,Val,230629PN031,NumberOfOrgans,1,"(1, 1, 1)",Prediction,1.000000
1,Segformer_bce_dice,Val,230629PN031,NumberOfOrgans,1,"(1, 1, 1)",expertized,1.000000
2,Segformer_bce_dice,Val,230629PN031,NumberOfOrgans,1,"(1, 1, 1)",before_expertized,1.000000
3,Segformer_bce_dice,Val,230629PN031,NumberOfOrgans,1,"(36, 47, 45)",Prediction,1.000000
4,Segformer_bce_dice,Val,230629PN031,NumberOfOrgans,1,"(36, 47, 45)",expertized,1.000000
...,...,...,...,...,...,...,...,...
209491,Unet_cldice_dice,Test,230629PN018,LateralRootLength,29,"(10, 14, 8)",expertized,125.584577
209492,Unet_cldice_dice,Test,230629PN018,LateralRootLength,29,"(10, 14, 8)",before_expertized,20.126268
209493,Unet_cldice_dice,Test,230629PN018,LateralRootLength,29,"(17, 23, 13)",Prediction,4.000000
209494,Unet_cldice_dice,Test,230629PN018,LateralRootLength,29,"(17, 23, 13)",expertized,119.834592


## normalized data frame

In [8]:
df_normalized = df_plant_wid.copy()
# Get the max value for each metric (regardless of time, model, or source)
max_value_for_each_metric = (
    df_normalized.groupby('metric')['value']
    .max()
    .reset_index()
    .rename(columns={'value': 'max_value'})
)

# Merge to assign max_value to each row
df_normalized = df_normalized.merge(max_value_for_each_metric, on='metric', how='left')
df_normalized['value'] = df_normalized['value'] / df_normalized['max_value']
df_normalized = df_normalized.drop(columns=['max_value'])
df_normalized

,model,split,box,metric,time,root_ids,source,value
0,Segformer_bce_dice,Val,230629PN031,NumberOfOrgans,1,"(1, 1, 1)",Prediction,0.034483
1,Segformer_bce_dice,Val,230629PN031,NumberOfOrgans,1,"(1, 1, 1)",expertized,0.034483
2,Segformer_bce_dice,Val,230629PN031,NumberOfOrgans,1,"(1, 1, 1)",before_expertized,0.034483
3,Segformer_bce_dice,Val,230629PN031,NumberOfOrgans,1,"(36, 47, 45)",Prediction,0.034483
4,Segformer_bce_dice,Val,230629PN031,NumberOfOrgans,1,"(36, 47, 45)",expertized,0.034483
...,...,...,...,...,...,...,...,...
209491,Unet_cldice_dice,Test,230629PN018,LateralRootLength,29,"(10, 14, 8)",expertized,0.037272
209492,Unet_cldice_dice,Test,230629PN018,LateralRootLength,29,"(10, 14, 8)",before_expertized,0.005973
209493,Unet_cldice_dice,Test,230629PN018,LateralRootLength,29,"(17, 23, 13)",Prediction,0.001187
209494,Unet_cldice_dice,Test,230629PN018,LateralRootLength,29,"(17, 23, 13)",expertized,0.035566


## Mean (not normalized) dataframe (mean for all plants)

In [9]:
# %% Agrégation par temps / model / metric
df_mean = (
    df_grouped
    .reset_index()  # remettre les colonnes
    .groupby(['model', 'metric', 'time', 'source'], as_index=False)
    .agg(mean_value=('value', 'mean'),
         std_value=('value', 'std')
         )
)

df_mean

,model,metric,time,source,mean_value,std_value
0,Segformer_bce,Convex_Area_Hull,1,Prediction,397.060906,67.128738
1,Segformer_bce,Convex_Area_Hull,1,before_expertized,390.886309,81.602448
2,Segformer_bce,Convex_Area_Hull,1,expertized,387.128963,39.024736
3,Segformer_bce,Convex_Area_Hull,2,Prediction,452.039324,72.314572
4,Segformer_bce,Convex_Area_Hull,2,before_expertized,443.585114,91.231965
...,...,...,...,...,...,...
4867,Unet_dice,TotalRootLength,28,before_expertized,1651.908930,916.730044
4868,Unet_dice,TotalRootLength,28,expertized,1817.790345,916.273673
4869,Unet_dice,TotalRootLength,29,Prediction,1768.862727,944.132276
4870,Unet_dice,TotalRootLength,29,before_expertized,1737.399024,960.907567


In [ ]:
all_models = sorted(df_mean['model'].unique().tolist())

# styles fixes par source (identiques pour tous les modèles)
source_styles = {
    'Prediction': dict(linestyle='-', marker='o'),
}
source_order = ['Prediction']


def plot_mean_all_models(metric_idx):
    metric = metrics[metric_idx]  # slider sur la métrique uniquement
    df_plot = df_mean[df_mean['metric'] == metric].copy()
    if df_plot.empty:
        print(f"Aucune donnée pour la métrique {metric}.")
        return

    # s'assurer que time est numérique et trié
    df_plot['time'] = pd.to_numeric(df_plot['time'], errors='coerce')
    df_plot = df_plot.sort_values(['model', 'source', 'time'])

    plt.figure(figsize=(14, 7))

    # tracer chaque modèle avec une couleur différente,
    # et pour chaque source un style différent
    for m in all_models:
        d_m = df_plot[df_plot['model'] == m]
        if d_m.empty:
            continue
        for src in source_order:
            d = d_m[d_m['source'] == src]
            if d.empty:
                continue
            st = source_styles.get(src, dict(linestyle='-', marker=None))
            # on utilise le cycle de couleurs de Matplotlib: une couleur par modèle
            plt.plot(
                d['time'], d['mean_value'],
                label=f"{m} – {src}",
                linewidth=2.0, markersize=6, alpha=0.95,
                **st
            )

    plt.title(f"Évolution de la valeur moyenne – métrique: {metric}")
    plt.xlabel('Time')
    plt.ylabel('Mean value')
    plt.grid(True, linestyle=':', alpha=0.5)

    # Légende compacte
    plt.legend(title='Model – Source', fontsize=9, ncol=2)
    plt.tight_layout()
    plt.show()


# widget: slider uniquement sur la métrique
widgets.interactive(
    plot_mean_all_models,
    metric_idx=metric_slider
)


interactive(children=(IntSlider(value=0, description='Metric', max=7), Output()), _dom_classes=('widget-intera…

In [ ]:
all_models = sorted(df_mean['model'].unique().tolist())

# styles fixes par source (identiques pour tous les modèles)
source_styles = {
    'Prediction': dict(linestyle='-', marker='o'),
}
source_order = ['Prediction']
colormap = plt.get_cmap("tab10")


def plot_mean_all_models(metric_idx):
    metric = metrics[metric_idx]  # slider sur la métrique uniquement
    df_plot = df_mean[df_mean['metric'] == metric].copy()
    if df_plot.empty:
        print(f"Aucune donnée pour la métrique {metric}.")
        return

    # s'assurer que time est numérique et trié
    df_plot['time'] = pd.to_numeric(df_plot['time'], errors='coerce')
    df_plot = df_plot.sort_values(['model', 'source', 'time'])

    plt.figure(figsize=(14, 7))

    # tracer chaque modèle avec une couleur différente,
    # et pour chaque source un style différent
    for m, style in zip(all_models, colormap.colors):
        d_m = df_plot[df_plot['model'] == m]
        if d_m.empty:
            continue
        for src in source_order:
            d = d_m[d_m['source'] == src]
            if d.empty:
                continue
            st = source_styles.get(src, dict(linestyle='-', marker=None))
            # on utilise le cycle de couleurs de Matplotlib: une couleur par modèle
            plt.plot(
                d['time'], d['mean_value'],
                label=f"{m}",
                linestyle=st['linestyle'], marker=st['marker'],
                color=style
            )
            # std
            plt.fill_between(
                d['time'], d['mean_value'] - d['std_value'],
                           d['mean_value'] + d['std_value'],
                alpha=0.2, color=style
            )

    plt.title(f"Évolution de la valeur moyenne – métrique: {metric}")
    plt.xlabel('Time')
    plt.ylabel('Mean value')
    plt.grid(True, linestyle=':', alpha=0.5)

    # Légende compacte
    plt.legend(title='Model – Source', fontsize=9, ncol=2)
    plt.tight_layout()
    plt.show()


# widget: slider uniquement sur la métrique
widgets.interactive(
    plot_mean_all_models,
    metric_idx=metric_slider
)

interactive(children=(IntSlider(value=0, description='Metric', max=7), Output()), _dom_classes=('widget-intera…

In [ ]:
# for each model and metric, plot the evolution over time of mean value

def plot_mean_evolution(model_idx, metric_idx):
    model = models[model_idx]
    metric = metrics[metric_idx]
    df_plot = df_mean[(df_mean['model'] == model) & (df_mean['metric'] == metric)]
    plt.figure(figsize=(16, 10))
    for source, style in zip(['Prediction', 'before_expertized', 'expertized'],
                             [{'color': 'red', 'linestyle': '-', 'marker': 'x'},
                              {'color': 'orange', 'linestyle': '--', 'marker': None},
                              {'color': 'green', 'linestyle': '-', 'marker': 'o'}]):
        df_s = df_plot[df_plot['source'] == source]
        if df_s.empty:
            continue
        plt.errorbar(
            df_s['time'], df_s['mean_value'],
            label=source,
            color=style['color'],
            linestyle=style['linestyle'],
            marker=style['marker'],
            linewidth=2.5,
            markersize=8,
            alpha=0.8
        )
        plt.fill_between(
            df_s['time'], df_s['mean_value'] - df_s['std_value'],
                          df_s['mean_value'] + df_s['std_value'],
            alpha=0.2, color=style['color']
        )
    plt.title(f"Mean evolution over time – {model} / {metric}")
    plt.xlabel('Time')
    plt.ylabel('Mean Value')
    plt.grid(True, linestyle=':', alpha=0.5)
    plt.legend(title='Source')
    plt.tight_layout()
    plt.show()


widgets.interactive(
    plot_mean_evolution,
    model_idx=model_slider,
    metric_idx=metric_slider
)



interactive(children=(IntSlider(value=0, description='Model', max=6), IntSlider(value=0, description='Metric',…

## Error

In [13]:
# Pivot the dataframe to get 'Prediction', 'expertized', and 'real' as columns
df_pivot = (
    df_plant_wid
    .pivot_table(
        index=['model', 'metric', 'time', 'split', 'box', 'root_ids'],
        columns='source',
        values='value'
    )
    .reset_index()
)

# Compute errors only if both columns exist
df_pivot['abs_error'] = (df_pivot['expertized'] - df_pivot['Prediction']).abs()
df_pivot['real_error'] = df_pivot['expertized'] - df_pivot['Prediction']

df_error = df_pivot[
    ['model', 'metric', 'time', 'split', 'box', 'root_ids', 'abs_error', 'real_error']
]
df_error

source,model,metric,time,split,box,root_ids,abs_error,real_error
0,Segformer_bce,Convex_Area_Hull,1,Test,230629PN008,"(8, 1, 1)",2.102307,-2.102307
1,Segformer_bce,Convex_Area_Hull,1,Test,230629PN008,"(16, 11, 8)",0.344508,0.344508
2,Segformer_bce,Convex_Area_Hull,1,Test,230629PN010,"(15, 1, 3)",0.066454,-0.066454
3,Segformer_bce,Convex_Area_Hull,1,Test,230629PN010,"(26, 16, 14)",0.693757,0.693757
4,Segformer_bce,Convex_Area_Hull,1,Test,230629PN010,"(44, 35, 32)",0.016828,-0.016828
...,...,...,...,...,...,...,...,...
69827,Unet_dice,TotalRootLength,29,Val,230629PN031,"(1, 1, 1)",129.536772,129.536772
69828,Unet_dice,TotalRootLength,29,Val,230629PN031,"(20, 19, 18)",497.480324,497.480324
69829,Unet_dice,TotalRootLength,29,Val,230629PN031,"(48, 47, 45)",134.087492,134.087492
69830,Unet_dice,TotalRootLength,29,Val,230629PN031,"(68, 66, 63)",3.381529,-3.381529


In [14]:
df_mean_error = (
    df_error
    .groupby(['model', 'metric', 'time'], as_index=False)
    .agg(
        abs_error_mean=('abs_error', 'mean'),
        real_error_mean=('real_error', 'mean'),
        std_abs_error=('abs_error', 'std'),
    )
)
df_mean_error

,model,metric,time,abs_error_mean,real_error_mean,std_abs_error
0,Segformer_bce,Convex_Area_Hull,1,12.555731,-9.931943,49.186977
1,Segformer_bce,Convex_Area_Hull,2,12.400503,-10.519181,48.641174
2,Segformer_bce,Convex_Area_Hull,3,18.788374,-4.594231,61.484383
3,Segformer_bce,Convex_Area_Hull,4,18.234050,-4.403006,61.262677
4,Segformer_bce,Convex_Area_Hull,5,18.436631,-4.040144,61.956750
...,...,...,...,...,...,...
1619,Unet_dice,TotalRootLength,25,93.149819,77.794890,100.333960
1620,Unet_dice,TotalRootLength,26,107.247941,93.108750,105.980646
1621,Unet_dice,TotalRootLength,27,123.805594,109.110001,116.809601
1622,Unet_dice,TotalRootLength,28,142.863203,127.876818,124.589044


In [ ]:
all_models = sorted(df_mean_error['model'].unique().tolist())


def plot_distance_all_models(metric_idx):
    metric = metrics[metric_idx]
    df_plot = df_mean_error[df_mean_error['metric'] == metric].copy()
    if df_plot.empty:
        print(f"Aucune donnée pour la métrique {metric}.")
        return

    df_plot['time'] = pd.to_numeric(df_plot['time'], errors='coerce')
    df_plot = df_plot.sort_values(['model', 'time'])

    plt.figure(figsize=(14, 7))
    for m in all_models:
        d = df_plot[df_plot['model'] == m]
        if d.empty:
            continue
        plt.plot(
            d['time'], d['abs_error_mean'],
            marker='o', linewidth=2.0, markersize=6, alpha=0.95, label=m
        )

        plt.fill_between(
            d['time'], d['abs_error_mean'] - d['std_abs_error'],
                       d['abs_error_mean'] + d['std_abs_error'],
            alpha=0.2
        )

    plt.title(f"Évolution de l'erreur absolue moyenne – métrique: {metric}")
    plt.xlabel('Time')
    plt.ylabel('Mean absolute error')
    plt.grid(True, linestyle=':', alpha=0.5)
    plt.legend(title='Model', ncol=2, fontsize=9)
    plt.tight_layout()
    plt.show()


# on garde le slider métrique existant
widgets.interactive(
    plot_distance_all_models,
    metric_idx=metric_slider
)


interactive(children=(IntSlider(value=0, description='Metric', max=7), Output()), _dom_classes=('widget-intera…

## Each model mean error normalization

In [16]:
df_max_value_for_each_metric = (
    df_mean_error
    .groupby(['metric'])['abs_error_mean']
    .max()
)
df_max_value_for_each_metric

metric
Convex_Area_Hull         941.830607
Intercept_curve_Area       2.709337
LateralRootLength        515.469565
NumberOfLateralRoots       7.581395
NumberOfOrgans             7.581395
PrimaryRootLength        499.424256
RootDensity                0.232979
TotalRootLength         1014.893821
Name: abs_error_mean, dtype: float64

In [17]:
#df_max_value_for_each_metric = (
#    df_mean_error
#    .groupby(['metric'])['abs_error_mean']
#    .max()
#)

df_max_value_for_each_metric = (
    df_mean[df_mean['source'] == 'expertized']
    .groupby('metric')['mean_value']
    .max()
)

df_max_value_for_each_metric

metric
Convex_Area_Hull        1703.636615
Intercept_curve_Area       5.440407
LateralRootLength       1065.244440
NumberOfLateralRoots      13.302326
NumberOfOrgans            14.302326
PrimaryRootLength        851.652623
RootDensity                1.055427
TotalRootLength         1916.897063
Name: mean_value, dtype: float64

In [18]:
df_normalized = df_mean_error.copy()
# Create a MultiIndex for mapping
df_normalized['max_value'] = df_normalized.set_index(['metric']).index.map(df_max_value_for_each_metric)
df_normalized['abs_error_mean'] = 1 - df_normalized['abs_error_mean'] / df_normalized['max_value']
df_normalized['real_error_mean'] = df_normalized['real_error_mean'] / df_normalized['max_value']
df_normalized['std_abs_error'] = df_normalized['std_abs_error'] / df_normalized['max_value']
df_normalized = df_normalized.drop(columns=['max_value'])

# rename abs_error_mean to normalized_abs_error_mean
df_normalized = df_normalized.rename(columns={'abs_error_mean': 'norm_abs_approx'})
df_normalized

,model,metric,time,norm_abs_approx,real_error_mean,std_abs_error
0,Segformer_bce,Convex_Area_Hull,1,0.992630,-0.005830,0.028872
1,Segformer_bce,Convex_Area_Hull,2,0.992721,-0.006175,0.028551
2,Segformer_bce,Convex_Area_Hull,3,0.988972,-0.002697,0.036090
3,Segformer_bce,Convex_Area_Hull,4,0.989297,-0.002584,0.035960
4,Segformer_bce,Convex_Area_Hull,5,0.989178,-0.002371,0.036367
...,...,...,...,...,...,...
1619,Unet_dice,TotalRootLength,25,0.951406,0.040584,0.052342
1620,Unet_dice,TotalRootLength,26,0.944051,0.048573,0.055288
1621,Unet_dice,TotalRootLength,27,0.935414,0.056920,0.060937
1622,Unet_dice,TotalRootLength,28,0.925472,0.066710,0.064995


In [ ]:
# %% Optionnel : visualisation de l'évolution de la distance moyenne au cours du temps

def plot_distance_evolution(model_idx, metric_idx):
    model = models[model_idx]
    metric = metrics[metric_idx]
    df_plot = df_normalized[(df_normalized['model'] == model) & (df_normalized['metric'] == metric)]
    if df_plot.empty:
        print("Aucune paire Prediction/expertized trouvée pour cette combinaison.")
        return
    plt.figure(figsize=(14, 7))
    plt.errorbar(
        df_plot['time'], df_plot['norm_abs_approx'],
        label='|Prediction - expertized| / max_{metric, time} (|Prediction - expertized|)',
        linestyle='-', marker='o', linewidth=2.5, markersize=7, alpha=0.9
    )
    plt.fill_between(
        df_plot['time'],
        df_plot['norm_abs_approx'] - df_plot['std_abs_error'],
        df_plot['norm_abs_approx'] + df_plot['std_abs_error'],
        alpha=0.2, color='blue'
    )
    plt.title(f"Évolution de l'erreur absolue normalisée – {model} / {metric}")
    plt.xlabel('Time')
    plt.ylabel('Norm. mean absolute error')
    plt.grid(True, linestyle=':', alpha=0.5)
    plt.legend()
    plt.ylim(0, 1.2)
    plt.tight_layout()
    plt.show()


widgets.interactive(
    plot_distance_evolution,
    model_idx=model_slider,
    metric_idx=metric_slider
)


interactive(children=(IntSlider(value=0, description='Model', max=6), IntSlider(value=0, description='Metric',…

In [ ]:
def plot_distance_all_models(metric_idx):
    metric = metrics[metric_idx]
    df_plot = df_normalized[df_normalized['metric'] == metric].copy()
    if df_plot.empty:
        print(f"Aucune donnée pour la métrique '{metric}'.")
        return

    # on s'assure que time est numérique et trié par modèle
    df_plot['time'] = pd.to_numeric(df_plot['time'], errors='coerce')
    df_plot = df_plot.sort_values(['model', 'time'])

    # couleurs auto (une par modèle)
    colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
    models_in_plot = df_plot['model'].unique().tolist()

    plt.figure(figsize=(14, 7))
    for i, m in enumerate(models_in_plot):
        d = df_plot[df_plot['model'] == m]
        if d.empty:
            continue

        # courbe + barres d'erreur (plus lisible que des aplats multiples)
        plt.errorbar(
            d['time'], d['norm_abs_approx'],
            yerr=d.get('std_abs_error', None),
            label=m,
            linestyle='-',
            marker='o',
            linewidth=2.0,
            markersize=6,
            alpha=0.9,
            color=colors[i % len(colors)],
            capsize=3,
        )

    plt.title(f"Évolution de l'erreur absolue normalisée — métrique: {metric}")
    plt.xlabel('Time')
    plt.ylabel('Norm. mean absolute error')
    plt.grid(True, linestyle=':', alpha=0.5)
    plt.legend(title='Model', ncol=2, fontsize=9)
    plt.ylim(0, 1)  # fixe l'ordonnée entre 0 et 1
    plt.tight_layout()
    plt.show()


# Slider uniquement pour la métrique
widgets.interactive(
    plot_distance_all_models,
    metric_idx=metric_slider
)

interactive(children=(IntSlider(value=0, description='Metric', max=7), Output()), _dom_classes=('widget-intera…

## Error at all time

In [21]:
df_norm_mean_at_all_time = df_normalized.groupby(['metric', 'model'])['norm_abs_approx'].mean()
df_norm_mean_at_all_time

metric                model             
Convex_Area_Hull      Segformer_bce         0.988253
                      Segformer_bce_dice    0.987522
                      Segformer_dice        0.988326
                      Unet_bce              0.987673
                      Unet_bce_dice         0.801951
                      Unet_cldice_dice      0.988182
                      Unet_dice             0.988556
Intercept_curve_Area  Segformer_bce         0.967278
                      Segformer_bce_dice    0.963729
                      Segformer_dice        0.967300
                      Unet_bce              0.970138
                      Unet_bce_dice         0.857167
                      Unet_cldice_dice      0.966812
                      Unet_dice             0.974637
LateralRootLength     Segformer_bce         0.948068
                      Segformer_bce_dice    0.940574
                      Segformer_dice        0.947187
                      Unet_bce              0.952414
     

In [22]:
all_models = sorted(df_normalized['model'].unique().tolist())
all_models = ['Segformer_bce', 'Segformer_bce_dice', 'Segformer_dice', 'Unet_bce', 'Unet_cldice_dice', 'Unet_dice',
              'Unet_bce_dice']
print(all_models)

['Segformer_bce', 'Segformer_bce_dice', 'Segformer_dice', 'Unet_bce', 'Unet_cldice_dice', 'Unet_dice', 'Unet_bce_dice']


In [23]:
import os
import pandas as pd


def extract_last_metrics(base_path):
    """
    Explore récursivement base_path pour trouver les fichiers CSV de scalars
    et construit un dictionnaire de dictionnaires : 
    {metric_name: {model_name: last_value}}
    """
    # conteneur principal : metric -> modèle -> valeur
    metrics_dict = {}

    # Parcours des répertoires de modèles
    for model_name in os.listdir(base_path):
        model_path = os.path.join(base_path, model_name, "logs", "scalars")
        if not os.path.isdir(model_path):
            continue  # on saute ce qui n'est pas un répertoire valide

        # Parcours des fichiers CSV de ce modèle
        for file in os.listdir(model_path):
            if not file.endswith(".csv"):
                continue

            file_path = os.path.join(model_path, file)
            try:
                df = pd.read_csv(file_path)
                if "value" not in df.columns or df.empty:
                    continue
                last_value = df["value"].iloc[-1]

                # Nettoyage du nom de la métrique
                metric_name = file.replace("val_", "").replace(".csv", "")

                # Initialisation si besoin
                if metric_name not in metrics_dict:
                    metrics_dict[metric_name] = {}

                metrics_dict[metric_name][model_name] = float(last_value)

            except Exception as e:
                print(f"⚠️ Impossible de lire {file_path}: {e}")

    return metrics_dict


base_path = os.path.expanduser("~/Documents/code/RSMLExtraction/Results/Training/Logs/")
cv_metrics_results = extract_last_metrics(
    base_path)  # {'Specificity': {'Segformer_bce_dice': 0.9985596538, 'Segformer_bce': 0.9988902807,
print(cv_metrics_results)


{'Specificity': {'Segformer_bce_dice': 0.9985596538, 'Segformer_bce': 0.9988902807, 'Unet_dice': 0.9992205501, 'Unet_bce_dice': 0.9992681742, 'Unet_bce': 0.999278307, 'Segformer_dice': 0.9986302853, 'Unet_cldice_dice': 0.9992388487}, 'HausdorffDistance': {'Segformer_bce_dice': 32.02287292, 'Segformer_bce': 32.4457283, 'Unet_dice': 30.6745224, 'Unet_bce_dice': 31.91540146, 'Unet_bce': 29.71125793, 'Segformer_dice': 31.13061523, 'Unet_cldice_dice': 29.60779953}, 'PersistenceWassersteinGPUParallel_0': {'Segformer_bce_dice': 3.07885313, 'Segformer_bce': 3.200774431, 'Unet_dice': 1.776620388, 'Unet_bce_dice': 2.485318661, 'Unet_bce': 1.714779496, 'Segformer_dice': 3.512531519, 'Unet_cldice_dice': 1.79317379}, 'Betti1JaccardRatioGPU': {'Segformer_bce_dice': 0.519500494, 'Segformer_bce': 0.4976301491, 'Unet_dice': 0.5835165381, 'Unet_bce_dice': 0.4768283963, 'Unet_bce': 0.5742316246, 'Segformer_dice': 0.5224539638, 'Unet_cldice_dice': 0.5835711956}, 'PixelAccuracy': {'Segformer_bce_dice': 0.9

In [24]:
df_cv_and_graph = (
    df_normalized
    .groupby(['metric', 'model'])[['norm_abs_approx', 'std_abs_error']]
    .mean()
    .reset_index()
)

#for metric, values_and_model in zip(metric_name, all_models_to_X):
#    df_cv_and_graph[metric] = df_cv_and_graph['model'].map(values_and_model)
for metric, model_values in cv_metrics_results.items():
    df_cv_and_graph[metric] = df_cv_and_graph['model'].map(model_values)
df_cv_and_graph

,metric,model,norm_abs_approx,std_abs_error,Specificity,HausdorffDistance,PersistenceWassersteinGPUParallel_0,Betti1JaccardRatioGPU,PixelAccuracy,AverageCenterlineDistance,...,Precision,Dice,train_batch_loss,Betti0VariationIndexGPU,Betti0RelativeErrorGPU,loss_BCEDiceLoss,Betti1RelativeErrorGPU,loss_BCE,loss_DiceLoss,loss_CLDice_Dice
0,Convex_Area_Hull,Segformer_bce,0.988253,0.035548,0.998890,32.445728,3.200774,0.497630,0.998045,0.681714,...,0.908470,0.914845,0.006781,0.180368,0.522483,NaN,3.515625e+06,0.195488,NaN,NaN
1,Convex_Area_Hull,Segformer_bce_dice,0.987522,0.035505,0.998560,32.022873,3.078853,0.519500,0.997703,0.750937,...,0.884037,0.901657,0.902534,0.185365,0.531169,0.788213,3.515625e+06,NaN,NaN,NaN
2,Convex_Area_Hull,Segformer_dice,0.988326,0.035543,0.998630,31.130615,3.512532,0.522454,0.997984,0.700577,...,0.891088,0.914666,0.234850,0.184165,0.465020,NaN,2.343750e+06,NaN,0.085334,NaN
3,Convex_Area_Hull,Unet_bce,0.987673,0.035397,0.999278,29.711258,1.714779,0.574232,0.998730,0.545730,...,0.940371,0.941333,0.004669,0.114866,0.221387,NaN,4.296875e+06,0.126959,NaN,NaN
4,Convex_Area_Hull,Unet_bce_dice,0.801951,0.118242,0.999268,31.915401,2.485319,0.476828,0.998328,0.676452,...,0.937094,0.926445,0.820538,0.348547,1.453146,0.763020,3.906252e+05,NaN,NaN,NaN
5,Convex_Area_Hull,Unet_cldice_dice,0.988182,0.035350,0.999239,29.607800,1.793174,0.583571,0.998770,0.529162,...,0.937405,0.941467,0.039700,0.118898,0.223939,NaN,1.056985e+07,NaN,NaN,0.03625
6,Convex_Area_Hull,Unet_dice,0.988556,0.035601,0.999221,30.674522,1.776620,0.583517,0.998733,0.552442,...,0.936090,0.941209,0.181497,0.102960,0.227172,NaN,2.734375e+06,NaN,0.058791,NaN
7,Intercept_curve_Area,Segformer_bce,0.967278,0.034716,0.998890,32.445728,3.200774,0.497630,0.998045,0.681714,...,0.908470,0.914845,0.006781,0.180368,0.522483,NaN,3.515625e+06,0.195488,NaN,NaN
8,Intercept_curve_Area,Segformer_bce_dice,0.963729,0.034564,0.998560,32.022873,3.078853,0.519500,0.997703,0.750937,...,0.884037,0.901657,0.902534,0.185365,0.531169,0.788213,3.515625e+06,NaN,NaN,NaN
9,Intercept_curve_Area,Segformer_dice,0.967300,0.036297,0.998630,31.130615,3.512532,0.522454,0.997984,0.700577,...,0.891088,0.914666,0.234850,0.184165,0.465020,NaN,2.343750e+06,NaN,0.085334,NaN


In [25]:
df_cv_and_graph_normalize_for_all_cv_metrics = df_cv_and_graph.copy()

for metric in cv_metrics_results.keys():
    if (df_cv_and_graph_normalize_for_all_cv_metrics[metric] > 1).any() or (
            df_cv_and_graph_normalize_for_all_cv_metrics[metric] < 0).any():
        max_value = df_cv_and_graph_normalize_for_all_cv_metrics[metric].max()
        df_cv_and_graph_normalize_for_all_cv_metrics[metric] = 1 - (
                df_cv_and_graph_normalize_for_all_cv_metrics[metric] / max_value
        )
df_cv_and_graph_normalize_for_all_cv_metrics

,metric,model,norm_abs_approx,std_abs_error,Specificity,HausdorffDistance,PersistenceWassersteinGPUParallel_0,Betti1JaccardRatioGPU,PixelAccuracy,AverageCenterlineDistance,...,Precision,Dice,train_batch_loss,Betti0VariationIndexGPU,Betti0RelativeErrorGPU,loss_BCEDiceLoss,Betti1RelativeErrorGPU,loss_BCE,loss_DiceLoss,loss_CLDice_Dice
0,Convex_Area_Hull,Segformer_bce,0.988253,0.035548,0.998890,0.000000,0.088756,0.497630,0.998045,0.681714,...,0.908470,0.914845,0.006781,0.180368,0.640447,NaN,0.667391,0.195488,NaN,NaN
1,Convex_Area_Hull,Segformer_bce_dice,0.987522,0.035505,0.998560,0.013033,0.123466,0.519500,0.997703,0.750937,...,0.884037,0.901657,0.902534,0.185365,0.634470,0.788213,0.667391,NaN,NaN,NaN
2,Convex_Area_Hull,Segformer_dice,0.988326,0.035543,0.998630,0.040533,0.000000,0.522454,0.997984,0.700577,...,0.891088,0.914666,0.234850,0.184165,0.679991,NaN,0.778261,NaN,0.085334,NaN
3,Convex_Area_Hull,Unet_bce,0.987673,0.035397,0.999278,0.084278,0.511811,0.574232,0.998730,0.545730,...,0.940371,0.941333,0.004669,0.114866,0.847650,NaN,0.593478,0.126959,NaN,NaN
4,Convex_Area_Hull,Unet_bce_dice,0.801951,0.118242,0.999268,0.016345,0.292442,0.476828,0.998328,0.676452,...,0.937094,0.926445,0.820538,0.348547,0.000000,0.763020,0.963043,NaN,NaN,NaN
5,Convex_Area_Hull,Unet_cldice_dice,0.988182,0.035350,0.999239,0.087467,0.489492,0.583571,0.998770,0.529162,...,0.937405,0.941467,0.039700,0.118898,0.845894,NaN,0.000000,NaN,NaN,0.03625
6,Convex_Area_Hull,Unet_dice,0.988556,0.035601,0.999221,0.054590,0.494205,0.583517,0.998733,0.552442,...,0.936090,0.941209,0.181497,0.102960,0.843669,NaN,0.741304,NaN,0.058791,NaN
7,Intercept_curve_Area,Segformer_bce,0.967278,0.034716,0.998890,0.000000,0.088756,0.497630,0.998045,0.681714,...,0.908470,0.914845,0.006781,0.180368,0.640447,NaN,0.667391,0.195488,NaN,NaN
8,Intercept_curve_Area,Segformer_bce_dice,0.963729,0.034564,0.998560,0.013033,0.123466,0.519500,0.997703,0.750937,...,0.884037,0.901657,0.902534,0.185365,0.634470,0.788213,0.667391,NaN,NaN,NaN
9,Intercept_curve_Area,Segformer_dice,0.967300,0.036297,0.998630,0.040533,0.000000,0.522454,0.997984,0.700577,...,0.891088,0.914666,0.234850,0.184165,0.679991,NaN,0.778261,NaN,0.085334,NaN


In [ ]:
df_scatter = df_cv_and_graph_normalize_for_all_cv_metrics.copy()

cv_metrics = list(cv_metrics_results.keys())
graph_metrics = df_scatter["metric"].unique().tolist()

models_order = df_scatter['model'].unique().tolist()
colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
color_map = {m: colors[i % len(colors)] for i, m in enumerate(models_order)}


def plot_scatter_norm_vs_external(cv_key, graph_key, show_error=True, alpha_pts=0.8, size_pts=40):
    if cv_key not in cv_metrics:
        print(f"Métrique '{cv_key}' inconnue. Choisis parmi: {cv_metrics}")
        return
    if graph_key not in graph_metrics:
        print(f"Métrique '{graph_key}' inconnue. Choisis parmi: {graph_metrics}")
        return
    d = df_scatter.dropna(subset=[cv_key, 'metric', 'norm_abs_approx']).copy()
    d = d[d['metric'] == graph_key]

    if d.empty:
        print("Nothing to see here")
        return

    plt.figure(figsize=(12, 7))
    for m in models_order:
        dm = d[d['model'] == m]
        if dm.empty:
            continue
        x = dm[cv_key].values
        y = dm['norm_abs_approx'].values
        c = color_map[m]

        plt.scatter(x, y, s=size_pts, alpha=alpha_pts, label=m, color=c)

        if show_error and 'std_abs_error' in dm.columns:
            yerr = dm['std_abs_error'].values
            plt.errorbar(x, y, yerr=yerr, fmt='none', ecolor=c, alpha=0.5, capsize=2)

    plt.title(f"norm_abs_approx {graph_key} vs {cv_key} — tous modèles")
    plt.xlabel(cv_key)
    plt.ylabel("norm_abs_approx")
    plt.grid(True, linestyle=":", alpha=0.5)
    plt.xlim(0, 1)
    plt.ylim(0, 1.1)
    plt.legend(title="Modèle", ncol=2, fontsize=9)
    plt.tight_layout()
    plt.show()


widgets.interactive(
    plot_scatter_norm_vs_external,
    cv_key=widgets.Dropdown(options=cv_metrics, value=cv_metrics[0], description="CV metric"),
    graph_key=widgets.Dropdown(options=graph_metrics, value="NumberOfOrgans", description="Graph error"),
    show_error=widgets.Checkbox(value=True, description="Barres d’erreur"),
    alpha_pts=widgets.FloatSlider(value=1.0, min=0.2, max=1.0, step=0.1, description="Alpha"),
    size_pts=widgets.IntSlider(value=40, min=10, max=120, step=5, description="Taille points"),
)

interactive(children=(Dropdown(description='CV metric', options=('Specificity', 'HausdorffDistance', 'Persiste…

In [27]:
df_radar = df_cv_and_graph_normalize_for_all_cv_metrics.copy()

# Axes
graph_metrics_all = sorted(df_radar["metric"].dropna().unique().tolist())
cv_metrics_all = sorted(cv_metrics_results.keys())

# Modèles + couleurs stables
models_all = df_radar['model'].dropna().unique().tolist()
colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
color_map = {m: colors[i % len(colors)] for i, m in enumerate(models_all)}


# --- Helpers radar ---
def _radar_angles(n):
    ang = np.linspace(0, 2 * np.pi, n, endpoint=False).tolist()
    return ang + ang[:1]


def _plot_poly(ax, angles, values, label, color, fill=False, alpha=0.18, marker=None):
    vals = list(values) + [values[0]]
    ax.plot(angles, vals, linewidth=2, color=color, label=label, marker=marker)
    if fill:
        ax.fill(angles, vals, alpha=alpha, color=color)


def _setup_ax(ax, labels, title):
    angles = _radar_angles(len(labels))
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(labels, fontsize=9)
    ax.set_yticks([0.25, 0.5, 0.75, 1.0])
    ax.set_yticklabels(["0.25", "0.5", "0.75", "1.0"], fontsize=8)
    ax.set_ylim(0, 1.0)
    ax.set_title(title, pad=12, fontsize=12)
    return angles


# --- Fonction de tracé ---
def plot_double_radar(models_to_show, graph_axes, cv_axes, fill_polys=True, show_legend=True):
    # Garde un ordre stable
    models = [m for m in models_all if m in set(models_to_show)]
    if len(models) == 0:
        print("Sélectionne au moins un modèle.")
        return

    graph_axes = [m for m in graph_metrics_all if m in set(graph_axes)]
    cv_axes = [m for m in cv_metrics_all if m in set(cv_axes)]
    if len(graph_axes) < 3 or len(cv_axes) < 3:
        print("Il faut au moins 3 axes par radar.")
        return

    # Prépare les valeurs par modèle
    # Radar gauche: pour chaque metric interne -> norm_abs_approx moyen (déjà en 0..1, plus haut = mieux)
    left_vals_by_model = {}
    for m in models:
        d = df_radar[df_radar['model'] == m][['metric', 'norm_abs_approx']]
        mp = dict(zip(d['metric'], d['norm_abs_approx']))
        left_vals_by_model[m] = [float(mp.get(ax, np.nan)) for ax in graph_axes]

    # Radar droit: métriques CV -> colonnes (scores 0..1, les 'error' déjà inversées plus haut)
    right_vals_by_model = {}
    for m in models:
        d = df_radar[df_radar['model'] == m].iloc[:1]  # valeurs CV identiques sur les lignes du modèle
        if d.empty:
            right_vals_by_model[m] = [np.nan] * len(cv_axes)
        else:
            row = d.iloc[0]
            right_vals_by_model[m] = [float(row[ax]) for ax in cv_axes]

    # Figure
    fig = plt.figure(figsize=(18, 10))
    ax_left = plt.subplot(1, 2, 1, projection='polar')
    ax_right = plt.subplot(1, 2, 2, projection='polar')

    # Axes / titres
    ang_left = _setup_ax(ax_left, graph_axes, "Graph metrics — norm_abs")
    ang_right = _setup_ax(ax_right, cv_axes, "CV metrics — score")

    # Un polygone par modèle sur chaque radar
    markers = ['o', 's', 'D', '^', 'v', 'X', 'P', '*']
    for i, m in enumerate(models):
        col = color_map[m]
        mrk = markers[i % len(markers)]
        # gauche
        lv = left_vals_by_model[m]
        _plot_poly(ax_left, ang_left, lv, label=m, color=col, fill=fill_polys, marker=mrk)
        # droite
        rv = right_vals_by_model[m]
        _plot_poly(ax_right, ang_right, rv, label=m, color=col, fill=fill_polys, marker=mrk)

    # Légende commune
    if show_legend:
        handles = [plt.Line2D([0], [0], marker=markers[i % len(markers)], color='w',
                              markerfacecolor=color_map[m], markersize=8, label=m)
                   for i, m in enumerate(models)]
        ax_right.legend(handles=handles, title="Modèles",
                        bbox_to_anchor=(1.25, 1.0), loc='upper left', fontsize=9)

    plt.tight_layout()
    plt.show()


# --- Widgets ---
w_models = widgets.SelectMultiple(options=models_all, value=tuple(models_all), description="Modèles",
                                  rows=min(8, len(models_all)))
w_graph = widgets.SelectMultiple(options=graph_metrics_all, value=tuple(graph_metrics_all), description="Graph axes",
                                 rows=min(10, len(graph_metrics_all)))
w_cv = widgets.SelectMultiple(options=cv_metrics_all, value=tuple(cv_metrics_all), description="CV axes",
                              rows=len(cv_metrics_all))
w_fill = widgets.Checkbox(value=True, description="Remplir")
w_legend = widgets.Checkbox(value=True, description="Légende")

widgets.interactive(
    plot_double_radar,
    models_to_show=w_models,
    graph_axes=w_graph,
    cv_axes=w_cv,
    fill_polys=w_fill,
    show_legend=w_legend
)

interactive(children=(SelectMultiple(description='Modèles', index=(0, 1, 2, 3, 4, 5, 6), options=('Segformer_b…